- models, prompts, and tools, think of chains as the pipeline that connects everything together into a workflow.
- A chain = sequence of steps where the output of one step becomes the input of the next.
- Instead of calling an LLM once, you compose multiple operations:


- With chains:
  - preprocess input
  - call multiple LLMs
  - format outputs
  - combine tools + memory

___
___

### Types of Chains (important)

___

- **1. Simple Chain (LLMChain style)**
  - Single prompt → single LLM

```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_template("Explain {topic} in simple terms")

llm = ChatOpenAI(model="gpt-4o-mini")

chain = prompt | llm

result = chain.invoke({"topic": "Kafka"})
print(result.content)

- This is the new LangChain Expression Language (LCEL) style
- `|` means: pipe output → next step

___

- **2.Sequential Chain (multi-step reasoning)**

- Step-by-step transformation
```python
chain = (
    ChatPromptTemplate.from_template("Explain {topic}") 
    | llm
    | ChatPromptTemplate.from_template("Summarize this: {text}")
    | llm
)

result = chain.invoke({"topic": "Kafka"})

___

- **3. Parallel Chain (RunnableParallel)**

- Run multiple tasks at same time
```python
from langchain_core.runnables import RunnableParallel

chain = RunnableParallel(
    definition = ChatPromptTemplate.from_template("Define {topic}") | llm,
    advantages = ChatPromptTemplate.from_template("Advantages of {topic}") | llm
)

result = chain.invoke({"topic": "Kafka"})
print(result)

___

- **4.Conditional Chain (branching logic)**
 

 Different paths based on input


```python
from langchain_core.runnables import RunnableBranch

def is_short(x):
    return len(x["topic"]) < 5

chain = RunnableBranch(
    (is_short, ChatPromptTemplate.from_template("Explain briefly {topic}") | llm),
    ChatPromptTemplate.from_template("Explain in detail {topic}") | llm
)

result = chain.invoke({"topic": "AI"})

___

- **5.Transform Chain (custom logic)**

Add Python logic inside pipeline

```python
from langchain_core.runnables import RunnableLambda

def add_exclamation(x):
    return {"topic": x["topic"] + "!!!"}

chain = (
    RunnableLambda(add_exclamation)
    | ChatPromptTemplate.from_template("Explain {topic}")
    | llm
)

result = chain.invoke({"topic": "Kafka"})
```

___